# 07 — Error Analysis and Model Failure Modes

This notebook diagnoses the final tuned Random Forest without changing the model. It studies prediction errors by band-gap range, uncertainty, crystal system, chemical system, and individual material, then saves reusable results for the project.

In [ ]:
import re, json, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from jarvis.db.figshare import data
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.models import create_model_pipeline
from src.preprocessing import FEATURE_COLUMNS
RESULTS, FIGURES = ROOT/'results', ROOT/'figures'
RESULTS.mkdir(exist_ok=True); FIGURES.mkdir(exist_ok=True)
print('Project root:', ROOT)


## 1. Load data and reproduce the official test split

The split matches Notebook 03: 80/20, `random_state=42`, stratified by zero/non-zero band gap. JARVIS IDs and formulas are attached for interpretation.

In [ ]:
df = pd.read_csv(ROOT/'data'/'enhanced_material_descriptors.csv')
jarvis_df = pd.DataFrame(data('dft_3d'))
assert len(df) == len(jarvis_df), 'Descriptor/JARVIS row mismatch.'
df = df.copy()
df.insert(0, 'jid', jarvis_df['jid'].astype(str).to_numpy())
df.insert(1, 'formula', jarvis_df['formula'].astype(str).to_numpy())

X, y = df[FEATURE_COLUMNS].copy(), pd.to_numeric(df['target_bandgap'], errors='coerce')
assert y.notna().all()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=(y > 0).astype(int))
test_meta = df.loc[X_test.index, ['jid','formula','crys','spg_number']].copy()
print('Train:', X_train.shape, 'Test:', X_test.shape)


## 2. Refit the final tuned model on the training partition

This uses the exact final configuration from Notebook 03. The resulting test metrics should reproduce MAE ≈ 0.2381 eV, RMSE ≈ 0.5559 eV, R² ≈ 0.8195.

In [ ]:
pipeline = create_model_pipeline()
pipeline.fit(X_train, y_train)
pred = pipeline.predict(X_test)
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)
print(f'MAE  : {mae:.4f} eV')
print(f'RMSE : {rmse:.4f} eV')
print(f'R²   : {r2:.4f}')


In [ ]:
analysis = test_meta.copy().reset_index(drop=True)
analysis['actual_bandgap'] = y_test.to_numpy()
analysis['predicted_bandgap'] = pred
analysis['residual'] = analysis['actual_bandgap'] - analysis['predicted_bandgap']
analysis['absolute_error'] = analysis['residual'].abs()
analysis.head()


## 3. Predicted vs actual

In [ ]:
plt.figure(figsize=(8,7))
plt.scatter(analysis.actual_bandgap, analysis.predicted_bandgap, s=12, alpha=0.35)
m = max(analysis.actual_bandgap.max(), analysis.predicted_bandgap.max())
plt.plot([0,m],[0,m],'--')
plt.xlabel('Actual band gap (eV)'); plt.ylabel('Predicted band gap (eV)')
plt.title('Predicted vs Actual Band Gap — Held-out Test Set')
plt.xlim(left=0); plt.ylim(bottom=0); plt.tight_layout()
plt.savefig(FIGURES/'error_predicted_vs_actual.png', dpi=300, bbox_inches='tight')
plt.show()


## 4. Residuals vs prediction

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(analysis.predicted_bandgap, analysis.residual, s=12, alpha=0.35)
plt.axhline(0, linestyle='--')
plt.xlabel('Predicted band gap (eV)'); plt.ylabel('Residual = actual − predicted (eV)')
plt.title('Residuals vs Predicted Band Gap'); plt.tight_layout()
plt.savefig(FIGURES/'error_residuals_vs_predicted.png', dpi=300, bbox_inches='tight')
plt.show()


## 5. Error by actual band-gap range

In [ ]:
def gap_group(v):
    if v == 0: return '0 eV'
    if v <= .5: return '0–0.5 eV'
    if v <= 1: return '0.5–1 eV'
    if v <= 2: return '1–2 eV'
    if v <= 3: return '2–3 eV'
    if v <= 5: return '3–5 eV'
    if v <= 10: return '5–10 eV'
    return '>10 eV'

analysis['bandgap_group'] = analysis.actual_bandgap.map(gap_group)
error_by_gap = analysis.groupby('bandgap_group', sort=False).agg(
    n=('absolute_error','size'), MAE_eV=('absolute_error','mean'),
    RMSE_eV=('residual', lambda x: np.sqrt(np.mean(x**2))),
    mean_actual_eV=('actual_bandgap','mean')).reset_index()
display(error_by_gap)

plt.figure(figsize=(9,5)); plt.bar(error_by_gap.bandgap_group, error_by_gap.MAE_eV)
plt.xlabel('Actual band-gap range'); plt.ylabel('MAE (eV)')
plt.title('Prediction Error by Actual Band-Gap Range'); plt.xticks(rotation=30)
plt.tight_layout(); plt.savefig(FIGURES/'error_by_bandgap_range.png', dpi=300, bbox_inches='tight'); plt.show()


## 6. Random Forest uncertainty proxy

For every test material we calculate the standard deviation of the 200 individual tree predictions. This is a relative uncertainty proxy, **not** a calibrated prediction interval.

In [ ]:
pre = pipeline.named_steps['preprocessor']; forest = pipeline.named_steps['model']
Xt = pre.transform(X_test)
tree_pred = np.vstack([tree.predict(Xt) for tree in forest.estimators_])
tree_mean = tree_pred.mean(axis=0); tree_std = tree_pred.std(axis=0)
print('Max tree-mean/pipeline difference:', np.max(np.abs(tree_mean-pred)))
analysis['prediction_uncertainty'] = tree_std

rho, p = spearmanr(analysis.prediction_uncertainty, analysis.absolute_error)
print(f'Spearman uncertainty/error correlation: {rho:.4f}')
print(f'p-value: {p:.4e}')


In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(analysis.prediction_uncertainty, analysis.absolute_error, s=12, alpha=0.35)
plt.xlabel('Random Forest uncertainty proxy (eV)'); plt.ylabel('Absolute prediction error (eV)')
plt.title('Prediction Error vs Random Forest Uncertainty'); plt.tight_layout()
plt.savefig(FIGURES/'error_vs_uncertainty.png', dpi=300, bbox_inches='tight'); plt.show()

analysis['uncertainty_quartile'] = pd.qcut(analysis.prediction_uncertainty, 4, labels=['Q1 lowest','Q2','Q3','Q4 highest'], duplicates='drop')
error_by_uncertainty = analysis.groupby('uncertainty_quartile', observed=True).agg(
    n=('absolute_error','size'), mean_uncertainty_eV=('prediction_uncertainty','mean'),
    MAE_eV=('absolute_error','mean'), RMSE_eV=('residual', lambda x: np.sqrt(np.mean(x**2)))
).reset_index()
display(error_by_uncertainty)


## 7. Error by crystal system

In [ ]:
crystal_error = analysis.groupby('crys', dropna=False).agg(
    n=('absolute_error','size'), MAE_eV=('absolute_error','mean'),
    RMSE_eV=('residual', lambda x: np.sqrt(np.mean(x**2))),
    mean_uncertainty_eV=('prediction_uncertainty','mean')).reset_index()
display(crystal_error.sort_values('MAE_eV', ascending=False).head(20))

plot_df = crystal_error[crystal_error.n >= 50].sort_values('MAE_eV')
plt.figure(figsize=(9,6)); plt.barh(plot_df.crys, plot_df.MAE_eV)
plt.xlabel('MAE (eV)'); plt.ylabel('Crystal system'); plt.title('Band-Gap Error by Crystal System (n ≥ 50)')
plt.tight_layout(); plt.savefig(FIGURES/'error_by_crystal_system.png', dpi=300, bbox_inches='tight'); plt.show()


## 8. Chemical-system error analysis

A chemical system is the set of unique elements in a formula, e.g. `LiMnO2 → Li-Mn-O`. This is a chemistry grouping, not a structural identity.

In [ ]:
formula_pattern = re.compile(r'([A-Z][a-z]?)')
def chemical_system(formula): return '-'.join(sorted(set(formula_pattern.findall(str(formula)))))
analysis['chemical_system'] = analysis.formula.map(chemical_system)
chemical_error = analysis.groupby('chemical_system').agg(
    n=('absolute_error','size'), MAE_eV=('absolute_error','mean'),
    RMSE_eV=('residual', lambda x: np.sqrt(np.mean(x**2))),
    mean_uncertainty_eV=('prediction_uncertainty','mean')).reset_index()
reliable_chemical = chemical_error[chemical_error.n >= 5].sort_values(['MAE_eV','n'], ascending=[False,False])
print('Test chemical systems:', len(chemical_error))
print('Systems with ≥5 test materials:', len(reliable_chemical))
display(reliable_chemical.head(20))


## 9. Worst individual predictions

In [ ]:
worst = analysis.sort_values('absolute_error', ascending=False).head(50).reset_index(drop=True)
worst_cols = ['jid','formula','crys','spg_number','chemical_system','actual_bandgap','predicted_bandgap','absolute_error','prediction_uncertainty']
worst = worst[worst_cols]
display(worst)
worst.to_csv(RESULTS/'worst_bandgap_predictions.csv', index=False)


## 10. Low-uncertainty but high-error cases

These are especially important: if uncertainty is low while error is high, the uncertainty proxy failed to flag a difficult prediction.

In [ ]:
analysis['uncertainty_rank'] = analysis.prediction_uncertainty.rank(pct=True)
analysis['error_rank'] = analysis.absolute_error.rank(pct=True)
analysis['uncertainty_error_gap'] = analysis.error_rank - analysis.uncertainty_rank
surprising = analysis.sort_values('uncertainty_error_gap', ascending=False).head(30)[[
    'jid','formula','chemical_system','actual_bandgap','predicted_bandgap','absolute_error','prediction_uncertainty','uncertainty_rank','error_rank'
]].reset_index(drop=True)
display(surprising)
surprising.to_csv(RESULTS/'low_uncertainty_high_error_cases.csv', index=False)


## 11. Save a reusable summary

The output is used for later interpretation and can be connected to the Streamlit Trustworthiness section.

In [ ]:
summary = {
  'test_set': {'n_materials': int(len(analysis)), 'mae_eV': float(mae), 'rmse_eV': float(rmse), 'r2': float(r2)},
  'uncertainty': {'method':'Random Forest tree prediction standard deviation', 'calibrated':False, 'spearman_error_correlation':float(rho), 'spearman_p_value':float(p)},
  'error_by_bandgap': error_by_gap.to_dict(orient='records'),
  'error_by_uncertainty': error_by_uncertainty.to_dict(orient='records'),
  'error_by_crystal_system': crystal_error.to_dict(orient='records'),
  'chemical_systems': {'total_test_systems':int(len(chemical_error)), 'systems_with_at_least_5_test_materials':int(len(reliable_chemical))},
  'outputs': ['results/worst_bandgap_predictions.csv','results/low_uncertainty_high_error_cases.csv']
}
with open(RESULTS/'error_analysis_summary.json','w',encoding='utf-8') as f: json.dump(summary,f,indent=4)
print('Saved error-analysis results.')


# Interpretation checklist

After running the notebook, answer these questions:

1. Does error increase for larger band gaps?
2. Does Random Forest uncertainty increase when absolute error increases?
3. Which crystal systems have unusually high error?
4. Which chemical systems have unusually high error?
5. Are there low-uncertainty/high-error failures?
6. Do these failures suggest missing structural information?

This notebook is deliberately diagnostic. It does **not** tune the model or change the official test result. Its purpose is to tell us what the next scientific improvement should address.